# Cue d'Etat — Pocket Detector Training

Trains a **YOLOv8n** model on a billiards dataset and exports **TFLite FP16**
ready to drop into the `PocketDetector` interface in the app.

**Runtime:** GPU — Runtime → Change runtime type → T4 GPU.

The dataset may include multiple classes (balls, table, pockets, etc.).
The model is trained on all classes as-is. You only need to know which
class index corresponds to pockets — set that in Section 2 and the
smoke-test and Android integration will use it correctly.

## 1. Install

In [ ]:
!pip install -q ultralytics roboflow
import ultralytics
ultralytics.checks()

## 2. Configuration

Fill these in, then run all cells.

In [ ]:
# ── Roboflow ─────────────────────────────────────────────────────
USE_ROBOFLOW       = True
ROBOFLOW_API_KEY   = "YOUR_API_KEY"
ROBOFLOW_WORKSPACE = "YOUR_WORKSPACE"
ROBOFLOW_PROJECT   = "YOUR_PROJECT"
ROBOFLOW_VERSION   = 1

# ── Pocket class ─────────────────────────────────────────────
# Run Section 3 first to see all class indices, then set this.
# If the dataset has multiple pocket classes (corner vs side),
# list all indices, e.g. [2, 3]
POCKET_CLASS_IDX = [0]   # <-- update after inspecting classes

# ── Training ───────────────────────────────────────────────
EPOCHS         = 100
IMGSZ          = 640
BATCH          = 16
CONF_THRESHOLD = 0.4   # minimum confidence shown in smoke-test

## 3. Load Dataset & Inspect Classes

In [ ]:
import yaml, os

if USE_ROBOFLOW:
    from roboflow import Roboflow
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    dataset = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT).version(ROBOFLOW_VERSION).download("yolov8")
    DATASET_YAML = dataset.location + "/data.yaml"
else:
    from google.colab import files
    import zipfile
    print("Upload your dataset zip (YOLOv8 format):")
    uploaded = files.upload()
    with zipfile.ZipFile(list(uploaded.keys())[0]) as z:
        z.extractall("/content/dataset")
    DATASET_YAML = "/content/dataset/data.yaml"

with open(DATASET_YAML) as f:
    meta = yaml.safe_load(f)

names = meta.get('names', [])
if isinstance(names, dict):
    names = [names[k] for k in sorted(names)]

print(f"Dataset: {DATASET_YAML}")
print(f"Classes ({meta['nc']}):")
for i, n in enumerate(names):
    print(f"  [{i}]  {n}")
print("\n→ Update POCKET_CLASS_IDX in Section 2 if needed, then continue.")

## 4. Train

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

results = model.train(
    data=DATASET_YAML,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=0,
    patience=20,
    name="pocket_detector",
    project="/content/runs",
    # Augmentation: pockets appear at all angles and in varied lighting
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    degrees=45.0, flipud=0.5, fliplr=0.5,
    mosaic=1.0, mixup=0.1,
)

print("Best weights:", results.save_dir)

## 5. Validate

In [ ]:
import os
best_weights = os.path.join(results.save_dir, "weights", "best.pt")
trained_model = YOLO(best_weights)
metrics = trained_model.val(data=DATASET_YAML)
print(f"mAP50:     {metrics.box.map50:.4f}")
print(f"mAP50-95:  {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall:    {metrics.box.mr:.4f}")
print()
if metrics.box.map50 > 0.85:
    print("✓ mAP50 > 0.85 — good to export")
else:
    print("⚠ mAP50 below 0.85 — consider more epochs or more data")

## 6. Export to TFLite FP16

In [ ]:
export_path = trained_model.export(
    format="tflite",
    imgsz=IMGSZ,
    half=True,
    int8=False,
    nms=True,
)
size_mb = os.path.getsize(export_path) / 1024 / 1024
print(f"TFLite: {export_path}  ({size_mb:.1f} MB)")

## 7. Smoke-test

Shows detections for the pocket class only (filtered by `POCKET_CLASS_IDX`).

In [ ]:
import numpy as np, cv2, glob, matplotlib.pyplot as plt, tensorflow as tf

interp = tf.lite.Interpreter(model_path=export_path)
interp.allocate_tensors()
inp_d = interp.get_input_details()
out_d = interp.get_output_details()
SZ    = int(inp_d[0]['shape'][1])
dtype = inp_d[0]['dtype']
print("Input:", inp_d[0]['shape'], dtype)

def infer(path):
    bgr = cv2.imread(path)
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    h, w = rgb.shape[:2]
    inp = np.expand_dims((cv2.resize(rgb, (SZ, SZ)) / 255.0).astype(dtype), 0)
    interp.set_tensor(inp_d[0]['index'], inp)
    interp.invoke()
    return rgb, interp.get_tensor(out_d[0]['index'])[0], w, h

yaml_dir = os.path.dirname(DATASET_YAML) + "/"
imgs = (glob.glob(yaml_dir + 'valid/images/*.jpg') +
        glob.glob(yaml_dir + 'valid/images/*.png'))[:4]

fig, axes = plt.subplots(1, max(len(imgs), 1), figsize=(5 * max(len(imgs), 1), 5))
if len(imgs) == 1: axes = [axes]

for ax, p in zip(axes, imgs):
    img, dets, w, h = infer(p)
    draw, n = img.copy(), 0
    for d in dets:
        # d = [x1, y1, x2, y2, conf, class_idx]  (with baked NMS)
        conf = float(d[4]) if len(d) >= 5 else 0
        cls  = int(d[5])   if len(d) >= 6 else 0
        if conf >= CONF_THRESHOLD and cls in POCKET_CLASS_IDX:
            x1,y1,x2,y2 = (int(d[0]*w/SZ), int(d[1]*h/SZ),
                            int(d[2]*w/SZ), int(d[3]*h/SZ))
            cv2.rectangle(draw, (x1,y1), (x2,y2), (0,255,0), 2)
            cv2.putText(draw, f"{conf:.2f}", (x1, y1-5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,0), 1)
            n += 1
    ax.imshow(draw); ax.set_title(f"{n} pockets"); ax.axis('off')

plt.tight_layout(); plt.show()

## 8. Download

In [ ]:
import shutil
from google.colab import files
out = "/content/pocket_detector.tflite"
shutil.copy(export_path, out)
print(f"pocket_detector.tflite  ({os.path.getsize(out)/1024/1024:.1f} MB)")
files.download(out)

## 9. Android Integration (v1.4)

1. Copy `pocket_detector.tflite` → `app/src/main/assets/`

2. `app/build.gradle.kts`:
```kotlin
implementation("org.tensorflow:tensorflow-lite:2.16.1")
implementation("org.tensorflow:tensorflow-lite-support:0.4.4")
```

3. Implement `PocketDetector` — filter output by the pocket class index from this notebook:
```kotlin
class TflitePocketDetector(context: Context) : PocketDetector {
    // Class indices that correspond to pockets in your dataset
    private val pocketClassIndices = setOf(0)  // match POCKET_CLASS_IDX above

    private val interpreter = Interpreter(
        FileUtil.loadMappedFile(context, "pocket_detector.tflite")
    )

    override fun detect(yBytes: ByteArray, width: Int, height: Int): List<PointF>? {
        // 1. Resize yBytes to 640x640, replicate Y channel to RGB
        // 2. Run interpreter
        // 3. Parse output [x1,y1,x2,y2,conf,class_idx]
        //    return centres where conf >= 0.4 && class_idx in pocketClassIndices
    }
}
```

4. Pass to `TableScanAnalyzer` in `ProtractorScreen.kt`:
```kotlin
val detector = remember { TflitePocketDetector(context) }
val tableScanAnalyzer = remember(tableScanViewModel) {
    TableScanAnalyzer(
        tableScanViewModel::onFrame,
        tableScanViewModel::onFeltColorSampled,
        pocketDetector = detector,
    )
}
```